# PAN Lines Theme Pipeline: set up this account

Builds the whole demo in this account from the files in this repository, then deploys the web app
into Snowflake. Run the cells in order, top to bottom. About 15 minutes and a few credits: the
synthetic notes, the first taxonomy and its gold labels are generated with live model calls.

| Step | What it does | Time |
|---|---|---|
| 1 | Database `PAN_DEMO_DB` and warehouse `PAN_DEMO_WH` | seconds |
| 2 | Preflight: both models answer, the grants are in place | ~20s |
| 3 | The pipeline: `sql/00` and `02` to `06` | ~1 min |
| 4 | The synthetic data (`sql/01`), then the first production config | ~4 min |
| 5 | The web app, built and deployed as an application service | ~6 min |

The account ends in the state the demo script starts from: taxonomy v001 in production, gold
labeled, nothing classified yet.

**Before you start**
- Open this notebook from the repository's `demo/` folder in a Workspace connected to the repo.
- Use the role that will own the demo (normally `SYSADMIN`) in the role picker.
- New account? Have an ACCOUNTADMIN run `demo/admin_setup.sql` once first (open it in this
  workspace and run it). Step 2 tells you if anything it grants is missing.

Rerunnable. Step 4 keeps data and a taxonomy it already generated, so running it again makes no
model calls and returns the account to the starting state; step 5 again redeploys the app at the
same URL.


In [ ]:
# Setup: find the repository and connect. Works in a Workspace notebook and on a local kernel
# (locally it uses SNOWFLAKE_CONNECTION_NAME, or your default CLI connection).
import sys
from pathlib import Path

for d in [Path.cwd(), *Path.cwd().parents]:
    if (d / "demo" / "setup_lib.py").exists():
        sys.path.insert(0, str(d / "demo"))
        break
else:
    raise FileNotFoundError("Open this notebook from the repository's demo/ folder.")

import setup_lib as setup

session = setup.get_session()
root = setup.repo_root()
print(f"Repository: {root}")

## 1. Database and warehouse

`PAN_DEMO_DB` and an XSMALL `PAN_DEMO_WH` (auto-suspends after a minute). The pipeline's dynamic
table, task and alert name this warehouse, so the build uses it throughout.

In [ ]:
setup.create_database_and_warehouse(session)

## 2. Preflight

Runs `demo/preflight.sql`. Calls both models the way the pipeline does (`claude-haiku-4-5`
classifies, `openai-gpt-5.2` labels gold and drafts revisions) and checks the grants the build
needs. A FAIL stops here with the fix; most are one line in `demo/admin_setup.sql`, for example
cross-region inference when a model is not hosted in this region.

In [ ]:
setup.preflight(session, root);

## 3. Build the pipeline

Runs `sql/00_setup.sql` and `02` to `06` as written, each with `EXECUTE IMMEDIATE FROM`.
`01_synthetic_data.sql` runs in step 4, after these, because its scorecard view reads tables that
`02` to `04` create.


In [ ]:
setup.build_pipeline(session, root)

## 4. Generate the demo data and the first production config

**Data.** Runs `sql/01_synthetic_data.sql`, which writes about 1000 synthetic nurse call notes
with `claude-haiku-4-5` (one call per encounter, under a minute) and fills the pipeline's three
input tables: the notes, the design sample and the gold set. Its answer key stays in
`SYNTHETIC_DATA`, which the pipeline never reads.

**First production config.** The same procedure calls as the README's bootstrap:
`PROPOSE_TAXONOMY` drafts v001 from the design sample with `openai-gpt-5.2`, `LOCK_TAXONOMY`
freezes it, `REGISTER_CONFIG` creates `haiku_enum_v1` on it, `EVALUATE_CONFIG` labels the 200
gold encounters with `openai-gpt-5.2` and scores the config on them, and `PROMOTE_CONFIG` puts
it in production. Then `demo/reset_webapp.sql` clears that evaluation and promotion so the app
starts clean.

Expected at the end: 1260 notes, 1000 encounters, v001 in production through `haiku_enum_v1`,
gold labels for v001 (about 280: some encounters get two), 0 classifications. The counts are the
same in every build because the scenarios are seeded, but the note text and the taxonomy are
generated, so the taxonomy's size, category names and measured accuracy differ from
the reference build the docs quote (one test build: 66 subthemes against the reference 78).

Rerunning keeps what is already there: notes in `NOTE_RAW` are not regenerated, and an existing
v001 with gold labels is not rebuilt. To start over, pass `regenerate=True` (new notes, which
invalidates the taxonomy's gold labels) and `rebuild=True` (new taxonomy and gold labels).


In [ ]:
counts = setup.generate_demo_data(session, root)
for k, v in counts.items():
    print(f"{k:22} {v}")

state = setup.bootstrap(session, root)
for k, v in state.items():
    print(f"{k:22} {v}")

## 5. Deploy the web app

Uploads `webapp/` to a stage, builds it in Snowflake (npm install and the Next.js build run in a
short-lived container, about 4 minutes), and creates the application service
`PAN_DEMO_DB.APP.PAN_THEME_PIPELINE` from the build. The app signs users in with Snowflake and
reads and writes only through the pipeline's views and procedures, as the service's owning role.

The service suspends after an hour without traffic and resumes on the next visit (the first page
after a resume takes a minute). Run this cell again after changing anything in `webapp/`.

The deployment settings (where the app lives, its warehouse, label and suspend time, what gets
uploaded, how it builds) come from `webapp/app.yml`, the app's manifest. With the Snowflake CLI
the same manifest deploys the app in one command, `snow app deploy`, run from `webapp/`; this cell
runs the same steps as SQL because a Workspace has no CLI.

In [ ]:
setup.upload_webapp(session, root)
setup.build_webapp(session)
app_url = setup.deploy_webapp_service(session, root)
print(f"\nApp: {app_url}")

## Done

Open the URL printed above and follow the demo script in `webapp/README.md`. The first page load
signs you in to Snowflake.

**Share the app.** Other users need USAGE on the database, schema and service, and the app runs
their requests as the service's owning role. For a role named `DEMO_VIEWER`:

```sql
GRANT USAGE ON DATABASE PAN_DEMO_DB TO ROLE DEMO_VIEWER;
GRANT USAGE ON SCHEMA PAN_DEMO_DB.APP TO ROLE DEMO_VIEWER;
GRANT USAGE ON APPLICATION SERVICE PAN_DEMO_DB.APP.PAN_THEME_PIPELINE TO ROLE DEMO_VIEWER;
```

**Between run-throughs**, reset to the starting state with the next cell (seconds, no model calls).

**Walkthrough notebook.** `demo/pan_lines_walkthrough.ipynb` takes one classification apart step by
step and runs against the same objects.

In [ ]:
# Reset between demo run-throughs: back to v001 in production, gold labeled, nothing run.
setup.run_sql_file(session, root, "demo/reset_webapp.sql")
setup.verify_state(session)

## Troubleshooting

- **Step 2 FAIL on a model**: the model is not hosted in this region. Uncomment the cross-region
  line in `demo/admin_setup.sql` and have an ACCOUNTADMIN run it.
- **Step 3 fails at `CREATE ALERT`**: the role lacks `EXECUTE ALERT`; see `demo/admin_setup.sql`.
- **Step 4 stops at `LOCK_TAXONOMY`**: the generated draft has a duplicate or reserved subtheme
  name (models don't always follow every rule); the message names it. Either draft a new one with
  `setup.bootstrap(session, root, rebuild=True)`, or fix the node with `EDIT_TAXONOMY_NODE` and
  make the remaining calls by hand (they are listed in the README's bootstrap).
- **Step 4 stops at `PROMOTE_CONFIG`**: the first evaluation did not pass the gate (accuracy below
  the 0.70 floor or too many errors). The message has the scores; rerun with `rebuild=True`.
- **Step 5 build fails**: the cell prints the build log. Builds download npm packages, so an
  account that has turned off default build egress needs an external access integration.
- **App URL shows an error after a while**: the service may be resuming; wait a minute and
  reload. The next cell shows its state and recent logs.

In [ ]:
# Service state and recent logs.
print(session.sql(f"DESCRIBE APPLICATION SERVICE {setup.APP_SERVICE}").collect()[0].as_dict())
print(session.sql(f"CALL SYSTEM$GET_APPLICATION_SERVICE_LOGS('{setup.APP_SERVICE}')").collect()[0][0][-3000:])